# Entregável 5 — Estratégia de Segmentação dos Sinais de ECG

**Disciplina:** Aquisição e Processamento de Biossinais  
**Equipe:** José Ferreira Lessa & Matheus Rocha Gomes da Silva  
**Orientador:** Prof. Dr. Victor Hugo C. de Albuquerque  
**Dataset:** PTB-XL — A Large Publicly Available Electrocardiography Dataset (PhysioNet)  
**Referência:** Wagner et al. (2020). PTB-XL, a large publicly available electrocardiography dataset. *Scientific Data*, 7(1), 154.  
**Data:** Março e Abril de 2026

---
---

## Objetivo

Este notebook descreve a etapa de **Segmentação (Janelamento)** dos sinais de ECG, responsável por transformar os registros contínuos de 10 segundos em unidades analíticas adequadas para a extração de atributos. Diferentemente de etapas anteriores — que avaliaram qualidade (Entregável 2), caracterizaram estatisticamente (Entregável 3) e limparam os sinais (Entregável 4) — aqui o foco é **estrutural**: definir *como* o sinal será dividido antes da extração de features.

O notebook está organizado nas seguintes seções:

1. **Fundamentação Teórica e Decisão de Projeto:** Revisão dos métodos de segmentação clássicos (janelas fixas, sobrepostas e orientadas por eventos fisiológicos), seguida da justificativa para a escolha da estratégia híbrida adotada neste trabalho.
2. **Instância Principal — Registro de 10 Segundos:** Carregamento dos sinais processados no Entregável 4, propagação de metadados e validação da integridade do dataset antes da segmentação.
3. **Segmentação por Batimento (Nível Morfológico):** Implementação do detector de pico R baseado em Pan-Tompkins, extração de janelas de 600 ms centradas no complexo QRS e processamento em lote do dataset completo.
4. **Validação da Estratégia de Segmentação:** Análise de estabilidade intra-janela (variância e Zero Crossing Rate por registro) e análise de variância inter-janela (variabilidade entre batimentos e entre classes diagnósticas), com inspeção visual das sobreposições.
5. **Exportação e Síntese:** Consolidação dos dois níveis de segmentação (registro e batimento), salvamento dos artefatos e discussão das implicações para a extração de atributos no Entregável 6.

> **Nota metodológica:** Esta etapa opera exclusivamente sobre os sinais classificados como **G (Excelente)** e **A (Aceitável)** após a reavaliação SQI do Entregável 4 (`sinais_bons_100hz.npy`). Sinais das classes P e U foram excluídos do pipeline principal, garantindo que apenas registros com integridade clínica comprovada alimentem as etapas subsequentes.

---

## 1. Importações, Configurações e Dependências

As bibliotecas utilizadas neste entregável são as mesmas dos entregáveis anteriores, sem adições externas. Bibliotecas adicionais específicas deste notebook:

- `gc` para liberação explícita de memória durante o processamento em lote de grandes arrays NumPy, evitando estouro de RAM.
- `tqdm.notebook` para barra de progresso adaptada ao ambiente Jupyter.

---

In [1]:
import os
import ast
import gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.signal as signal
import scipy.stats as stats
from pathlib import Path
from tqdm.notebook import tqdm
from IPython.display import display, Markdown
import warnings

warnings.filterwarnings('ignore', category=UserWarning)

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

np.random.seed(42)

## 2. Configurações Globais

Aqui são definidas as constantes globais do projeto, já apresentadas e justificadas nos entregáveis anteriores.

---

In [2]:
FS = 100
N_LEADS = 12
LEAD_NAMES = ['I', 'II', 'III', 'aVL', 'aVR', 'aVF', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6']
FOLDS_TREINO = [1, 2, 3, 4, 5, 6, 7, 8]
FOLD_VAL = 9
FOLD_TEST = 10

DIR_OUT_D4 = Path('../../entregavel-4/outputs/')
FIGS_DIR   = Path('../figuras/')
OUT_DIR    = Path('../outputs/')

for d in [FIGS_DIR, OUT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print('Configuração concluída.')
print(f'Figuras em : {FIGS_DIR.resolve()}')
print(f'Outputs em : {OUT_DIR.resolve()}')

Configuração concluída.
Figuras em : C:\Users\josel\OneDrive\Documentos\Biossinais\entregaveis\entregavel-5\figuras
Outputs em : C:\Users\josel\OneDrive\Documentos\Biossinais\entregaveis\entregavel-5\outputs


## 3. Carregamento dos Sinais Processados (Entregável 4)

Partimos dos arquivos gerados ao final do Entregável 4, que contêm exclusivamente os sinais classificados como **G (Excelente)** e **A (Aceitável)** após a reavaliação SQI pós-limpeza:

- `sinais_bons_100hz.npy` — array NumPy de shape `(N, 1000, 12)`, com os sinais filtrados e winsorizados, prontos para análise;
- `ptbxl_bons_com_sqi.csv` — metadados enriquecidos (SQI, superclasses, splits, etc.) associados a esses registros.

A coluna `superclasses_clean` é recuperada via `ast.literal_eval`, uma vez que foi serializada como string durante o salvamento em CSV.

---

In [3]:
npy_path = DIR_OUT_D4 / 'sinais_bons_100hz.npy'
csv_path = DIR_OUT_D4 / 'ptbxl_bons_com_sqi.csv'

if not npy_path.exists():
    raise FileNotFoundError(f"Arquivo não encontrado: {npy_path}\nExecute o Entregável 4 primeiro.")

print("Carregando sinais limpos e metadados...")
sinais = np.load(str(npy_path))
df = pd.read_csv(str(csv_path), index_col='ecg_id')
df['superclasses_clean'] = df['superclasses_clean'].apply(ast.literal_eval)
df['recording_date']     = pd.to_datetime(df['recording_date'], errors='coerce')

print(f"Dataset carregado: {sinais.shape[0]} registros × {sinais.shape[1]} amostras × {sinais.shape[2]} derivações.")
print(f"Colunas de metadados disponíveis: {df.shape[1]}")

FileNotFoundError: Arquivo não encontrado: ..\..\entregavel-4\outputs\sinais_bons_100hz.npy
Execute o Entregável 4 primeiro.

---
---
## Seção 1 — Fundamentação Teórica e Decisão de Projeto

---

### 1.1 Métodos de Segmentação em Sinais de ECG

A **segmentação** — ou janelamento — é a etapa que transforma um sinal contínuo em instâncias discretas, cada uma constituindo uma unidade de análise independente. A escolha do método de segmentação tem impacto direto na quantidade de instâncias geradas, na representatividade das features extraídas e na validade estatística das análises subsequentes.

Na literatura de processamento de biossinais, três abordagens principais são utilizadas:

---

#### A. Janelas Fixas com Sobreposição (*Sliding Window*)

Consiste em dividir o sinal em segmentos de duração fixa $T_w$, avançando com um passo $\Delta t$ — frequentemente menor que $T_w$, gerando sobreposição entre janelas consecutivas.

$$\text{N\_janelas} = \left\lfloor \frac{T_{total} - T_w}{\Delta t} \right\rfloor + 1$$

**Quando usar:** amplamente adotada em abordagens de *Deep Learning* (CNN, LSTM) para aumentar o volume de dados de treino. Especialmente eficaz quando o modelo aprende representações internas sem intervenção explícita de engenharia de features.

**Limitação:** em abordagens de ML clássico com features manuais, a sobreposição introduz **redundância artificial** e **correlação entre amostras**, podendo inflar métricas de validação cruzada e violar o pressuposto de independência amostral.

---

#### B. Janelas Fixas sem Sobreposição (*Non-overlapping Windows*)

Variante da anterior com $\Delta t = T_w$, produzindo janelas disjuntas e, portanto, estatisticamente independentes.

**Quando usar:** preferível para extração de features clássicas (domínio do tempo, frequência e não lineares) quando se deseja preservar a independência entre instâncias.

**Limitação:** para registros curtos (ex: 10 segundos), o número de instâncias por registro é pequeno.

---

#### C. Segmentação por Eventos Fisiológicos (*Event-based Segmentation*)

Em vez de dividir o sinal em intervalos temporais arbitrários, esta abordagem alinha as janelas a **eventos cardíacos** detectados, como o pico R do complexo QRS. Cada batimento torna-se uma instância independente, com duração variável ou normalizada.

**Quando usar:** ideal para extração de features morfológicas (forma das ondas P, QRS, T) e métricas de variabilidade de intervalo RR (HRV). Reflete de forma mais fiel a estrutura fisiológica do ECG.

**Limitação:** depende da qualidade do detector de batimentos, sendo sensível a artefatos e arritmias complexas.

---

**Referências:**

- [UnderstandingDigitalSignalProcessing](../../../docs/research/UnderstandingDigitalSignalProcessing.pdf)
- [Pan, J. & Tompkins, W.J. (1985)](../../../docs/research/PAN_TOMPKINS.pdf)


### 1.2 Decisão de Projeto: Estratégia Híbrida

Com base na análise dos métodos apresentados, foi adotada uma **estratégia híbrida em dois níveis**, cuja justificativa é apresentada a seguir.

---

#### Nível 1 — Instância Principal: Registro de 10 Segundos (Opção B)

**Justificativa clínica:**
O PTB-XL é estruturado em torno da unidade de registro de 10 segundos, que corresponde ao exame clínico padrão de ECG de 12 derivações. Essa janela foi projetada para capturar pelo menos um ciclo cardíaco completo em condições normais ($\approx$ 60–100 bpm) e contemplar features globais como energia espectral por banda, métricas de HRV de curto prazo e estatísticas de forma.

**Justificativa estatística:**
A preservação do registro de 10 segundos como unidade primária garante:

- **Independência entre amostras:** cada instância corresponde a um exame distinto, evitando correlação artificial;
- **Ausência de *label inflation*:** a estratégia não multiplica artificialmente o número de registros, preservando a integridade dos labels diagnósticos;
- **Consistência com a divisão estratificada do dataset:** os folds de treino/validação/teste definidos no Entregável 1 operam sobre esse nível, assegurando que todas as análises subsequentes sejam comparáveis.

---

#### Nível 2 — Segmentação Auxiliar: Janelas por Batimento (Opção C)

**Justificativa clínica:**
Determinadas classes de features — especialmente aquelas relacionadas à **morfologia das ondas cardíacas** (duração do QRS, amplitude da onda T, padrão da onda P) e à **variabilidade de intervalo RR (HRV)** — não podem ser extraídas de forma confiável sobre o registro completo de 10 segundos. A segmentação por batimento permite calcular essas métricas individualmente e, em seguida, **agregá-las** (média, desvio padrão, mínimo, máximo) ao nível do registro.

**Justificativa metodológica:**
Ao manter as features morfológicas no nível do registro (via agregação), preservamos a consistência com a instância principal. Dessa forma, o produto final desta etapa não é um dataset com mais linhas, mas sim um conjunto enriquecido de features por registro.

---

#### Síntese da Decisão

| Nível | Janela | Sobreposição | Finalidade | Produto no E6 |
|---|---|---|---|---|
| Registro | 10 s (1000 amostras) | Não | Features globais (energia, HRV, estatísticas) | 1 linha por registro |
| Batimento | 600 ms (60 amostras) | N/A (eventos) | Features morfológicas e de variabilidade | Agregado ao registro |

Essa arquitetura é detalhada nas próximas seções.

---

---
---
## Seção 2 — Instância Principal: Registro de 10 Segundos

Nesta seção, formalizamos o conjunto de instâncias de nível superior, verificamos a integridade dos dados carregados e realizamos a propagação dos metadados necessários para o Entregável 6.

---

### 2.1 Propagação de Metadados e Verificação de Integridade

Cada registro de 10 segundos constitui uma linha independente no dataset final. O objetivo aqui é garantir que o índice dos sinais (`sinais[i]`) corresponda exatamente ao `ecg_id` na linha `i` do DataFrame, preservando a rastreabilidade entre sinal e metadados.

Além disso, verificamos a distribuição dos registros por fold e por superclasse diagnóstica, assegurando que o conjunto mantém o balanceamento esperado após a exclusão dos registros classe U no Entregável 4.

---

In [ ]:
# Colunas de metadados essenciais para o pipeline
colunas_meta = ['patient_id', 'strat_fold', 'quality_class',
                'superclasses_clean', 'n_superclasses', 'split']

colunas_meta = [c for c in colunas_meta if c in df.columns]

df_instancias = df[colunas_meta].copy()

# Verificação de integridade: quantidade de sinais deve ser igual à de linhas
assert sinais.shape[0] == len(df_instancias), (
    f"Divergência: {sinais.shape[0]} sinais vs {len(df_instancias)} linhas no DataFrame"
)

# Helper para primeira superclasse (ou NONE)
def get_primary_class(x):
    if isinstance(x, list) and len(x) > 0:
        return x[0]
    return 'NONE'

df_instancias['primary_class'] = df_instancias['superclasses_clean'].apply(get_primary_class)

display(Markdown(f"""
**Integridade verificada.**
- Total de registros: **{len(df_instancias)}**
- Shape do array de sinais: **{sinais.shape}** (registros × amostras × derivações)
"""))

# Distribuição por fold
display(Markdown("**Distribuição de Registros por Fold:**"))
display(df_instancias['strat_fold'].value_counts().sort_index())

# Distribuição por superclasse primária
display(Markdown("**Distribuição por Superclasse Primária:**"))
display(df_instancias['primary_class'].value_counts())

**Comentários sobre a subseção 2.1:**

[Inserir comentário após análise das tabelas geradas — verificar se a distribuição por fold e por superclasse é consistente com os folds definidos no Entregável 1 e se a exclusão dos registros U gerou algum desbalanceamento expressivo entre as classes diagnósticas.]

---

### 2.2 Visualização Panorâmica dos Sinais (15 Exemplos)

Para garantir que o carregamento foi bem-sucedido e que os sinais apresentam a morfologia esperada após a limpeza do Entregável 4, são visualizados 3 registros de cada uma das 5 superclasses diagnósticas principais na derivação DII.

Esta visualização serve também como referência qualitativa para comparar padrões morfológicos entre classes — diferenças na forma e amplitude já identificadas nos Entregáveis 1 e 3 devem ser observáveis aqui, agora sobre sinais limpos.

---

In [ ]:
classes_target = ['NORM', 'MI', 'CD', 'STTC', 'HYP']

fig, axes = plt.subplots(len(classes_target), 3, figsize=(18, 15), sharex=True)
t_eixo = np.arange(1000) / FS

for i, cls in enumerate(classes_target):
    ids_cls = df_instancias[df_instancias['primary_class'] == cls].index[:3]

    for j in range(3):
        ax = axes[i, j]
        if j < len(ids_cls):
            eid = ids_cls[j]
            pos = df_instancias.index.get_loc(eid)
            ax.plot(t_eixo, sinais[pos, :, 1], color='black', lw=0.7)
            ax.set_title(f'{cls} — ID {eid}', fontsize=9)
        else:
            ax.text(0.5, 0.5, 'N/A', ha='center', va='center', transform=ax.transAxes)
            ax.set_axis_off()

        if j == 0:
            ax.set_ylabel('mV')
        if i == len(classes_target) - 1:
            ax.set_xlabel('Tempo (s)')

plt.suptitle('Painel Panorâmico — Derivação DII (15 exemplos, sinais limpos)', fontsize=14)
plt.tight_layout()
fig.savefig(FIGS_DIR / 'painel_15_exemplos.png', dpi=150, bbox_inches='tight')
plt.show()

**Comentários sobre a subseção 2.2:**

[Inserir comentário após análise dos gráficos — verificar se as diferenças morfológicas esperadas entre as classes (HYP com maior amplitude, MI com alterações de ST, AFIB com ritmo irregular capturado via CD/STTC) são visíveis após a limpeza. Comentar sobre a qualidade visual dos sinais pós-filtragem em comparação aos sinais brutos do Entregável 1.]

---

---
---
## Seção 3 — Segmentação por Batimento (Nível Morfológico)

Nesta seção, implementamos o algoritmo de detecção de picos R e extraímos janelas de batimento centradas em cada complexo QRS identificado. O produto desta seção é um array tridimensional `(N_batimentos × 60 × 12)` que alimentará, após agregação, as features morfológicas do Entregável 6.

---

### 3.1 Fundamentação: Algoritmo Pan-Tompkins

O detector de picos R implementado segue a lógica do **algoritmo Pan-Tompkins** (1985), que é o método de referência para detecção de ondas R em sinais de ECG e tem sido amplamente utilizado em trabalhos de processamento de biossinais.

O algoritmo opera em cinco etapas sequenciais:

**1. Filtragem Passa-Banda (5–15 Hz):**
Isola a faixa de frequência característica do complexo QRS, atenuando componentes de baixa frequência (ondas P e T, drift) e de alta frequência (ruído muscular). Essa etapa preserva a energia principal do QRS, que concentra a maior parte de sua energia na faixa de 10–20 Hz.

**2. Derivada e Quadratura:**
A derivada do sinal filtrado $\frac{dy}{dt}$ enfatiza as inclinações acentuadas características do complexo QRS. A quadratura $(dy/dt)^2$ torna todos os valores positivos e amplifica transições rápidas de amplitude.

$$s[n] = \left( y[n] - y[n-1] \right)^2$$

**3. Integração por Janela Móvel (150 ms):**
Suaviza múltiplos picos gerados na etapa anterior, resultando em uma envoltória de energia que facilita a detecção de eventos. A largura da janela é escolhida como uma estimativa da duração típica do QRS.

**4. Detecção de Picos e Período Refratário:**
Identifica máximos locais acima de um limiar de proeminência, com um período refratário mínimo de 200 ms (~300 bpm), que evita detecções duplas para um mesmo batimento.

**5. Refratário Adaptativo:**
Após uma primeira detecção conservadora (200 ms, equivalente a 300 bpm), o período refratário
é recalibrado para 60% do intervalo RR mediano estimado, com piso de 150 ms. Isso permite
acomodar taquicardias sem aumentar o risco de detecções duplas em ritmos normais.

A janela de batimento extraída abrange **200 ms pré-pico R** e **400 ms pós-pico R**, totalizando 600 ms (60 amostras a 100 Hz), capturando integralmente as ondas P, QRS e T do batimento.

**Referência:** 

- [Pan, J. & Tompkins, W.J. (1985)](../../../docs/research/PAN_TOMPKINS.pdf)

---

### 3.2 Implementação do Detector e Extrator de Janelas

---

In [ ]:
def detector_pan_tompkins(sinal_lead, fs=100):
    """
    Detecta picos R via algoritmo Pan-Tompkins (versão simplificada).

    Parâmetros:
    - sinal_lead : array 1D com o sinal de uma derivação
    - fs         : frequência de amostragem (Hz)

    Retorna:
    - picos : índices dos picos R detectados
    """
    nyq = 0.5 * fs

    # 1. Passa-banda 5–15 Hz
    b, a = signal.butter(2, [5 / nyq, 15 / nyq], btype='bandpass')
    filtrado = signal.filtfilt(b, a, sinal_lead)

    # 2. Derivada e quadratura
    derivada = np.diff(filtrado)
    quadrado = derivada ** 2

    # 3. Janela móvel de 150 ms
    n_janela = int(0.15 * fs)
    integrado = np.convolve(quadrado, np.ones(n_janela) / n_janela, mode='same')

    # 4. Limiar de prominência baseado em percentil — robusto a picos espúrios
    # np.max() seria sensível a um único artefato de alta amplitude;
    # o percentil 95 representa o topo da distribuição real do sinal integrado.
    proeminencia_min = np.percentile(integrado, 95) * 0.30
    # verificar se está funcionando bem para o dataset

    # 5. Período refratário adaptativo
    # Primeira passagem com refratário conservador (200 ms = 300 bpm)
    dist_inicial = int(0.20 * fs)
    picos_brutos, _ = signal.find_peaks(integrado, distance=dist_inicial,
                                         prominence=proeminencia_min)

    # Se houver picos suficientes, estima o RR mediano e ajusta o refratário
    # para 60% do RR mediano (mínimo de 150 ms para evitar detecções duplas)
    if len(picos_brutos) >= 2:
        rr_mediano = np.median(np.diff(picos_brutos))
        dist_min = max(int(rr_mediano * 0.60), int(0.15 * fs))
    else:
        dist_min = dist_inicial

    # Segunda passagem com refratário refinado
    picos, _ = signal.find_peaks(integrado, distance=dist_min,
                                  prominence=proeminencia_min)
    return picos


def extrair_janelas_beat(sig_12_leads, picos, fs=100):
    """
    Extrai janelas de 600 ms centradas em cada pico R.

    Janela: 200 ms pré-pico + 400 ms pós-pico = 60 amostras a 100 Hz.

    Parâmetros:
    - sig_12_leads : array (N_amostras × 12)
    - picos        : índices dos picos R
    - fs           : frequência de amostragem (Hz)

    Retorna:
    - janelas : array (N_beats × 60 × 12)
    - metadados : lista de dicionários por batimento
      Cada dicionário inclui 'aviso_sobreposicao' (bool) e 'rr_min_ms' (float):
      - aviso_sobreposicao=True indica que o RR mínimo do registro é menor que
        a janela de 600 ms, com risco de a onda T do batimento anterior invadir
        o início da janela atual (relevante para taquicardias).

    """
    pre  = int(0.20 * fs)   # 20 amostras
    post = int(0.40 * fs)   # 40 amostras
    n_amostras = sig_12_leads.shape[0]

    janelas, metadados = [], []

    # Calcula o RR mínimo do registro para flag de sobreposição
    if len(picos) >= 2:
        rr_min_ms = np.min(np.diff(picos)) * (1000 / fs)
        aviso_sobreposicao = rr_min_ms < (pre + post) * (1000 / fs)  # < 600 ms
    else:
        rr_min_ms = np.nan
        aviso_sobreposicao = False

    for k, r_idx in enumerate(picos):
        # Garante que a janela não ultrapassa os limites do sinal
        if r_idx < pre or r_idx > (n_amostras - post):
            continue

        janela = sig_12_leads[r_idx - pre : r_idx + post, :]
        rr_ms  = ((picos[k + 1] - r_idx) * (1000 / fs)
                  if k < len(picos) - 1 else np.nan)

        janelas.append(janela)
        metadados.append({
            'r_peak_idx'         : int(r_idx),
            'rr_interval_ms'     : rr_ms,
            'aviso_sobreposicao' : aviso_sobreposicao,
            'rr_min_ms'          : round(rr_min_ms, 1) if not np.isnan(rr_min_ms) else np.nan
        })

    if len(janelas) == 0:
        return np.empty((0, pre + post, 12), dtype=np.float32), []

    return np.array(janelas, dtype=np.float32), metadados


print("Funções de detecção e extração definidas.")

Funções de detecção e extração definidas.


### 3.3 Processamento em Lote

A extração de batimentos é realizada para todos os registros do dataset, utilizando a **derivação DII (Lead II)** como referência para o detector Pan-Tompkins — prática padrão na literatura, pois essa derivação costuma apresentar o complexo QRS com a melhor relação sinal-ruído e amplitude mais elevada na maioria dos pacientes.

Após a extração, os sinais de 10 segundos são explicitamente liberados da memória (`del sinais`) para evitar estouro de RAM durante o processamento, que opera sobre um array de grande porte.

---

In [4]:
lista_janelas   = []
lista_meta_bt   = []

display(Markdown(f"Processando **{len(df_instancias)}** registros..."))

for i, eid in enumerate(tqdm(df_instancias.index, desc='Extraindo batimentos')):
    sig = sinais[i]  # (1000 × 12)

    picos = detector_pan_tompkins(sig[:, 1], FS)   # Lead II como referência
    janelas, meta = extrair_janelas_beat(sig, picos, FS)

    if len(janelas) == 0:
        continue

    lista_janelas.append(janelas)

    cls_primaria = df_instancias.loc[eid, 'primary_class']
    fold         = df_instancias.loc[eid, 'strat_fold']
    split        = df_instancias.loc[eid, 'split'] if 'split' in df_instancias.columns else 'unknown'

    for m in meta:
        m.update({
            'ecg_id'        : eid,
            'patient_id'    : df.loc[eid, 'patient_id'],
            'strat_fold'    : fold,
            'split'         : split,
            'primary_class' : cls_primaria
        })
        lista_meta_bt.append(m)

# Libera a memória dos sinais de 10 s
# Guard contra reexecução fora de ordem: se a célula rodar novamente após
# o array já ter sido deletado, o 'del' causaria NameError sem o guard.
if 'sinais' in dir():
    print("Liberando memória dos sinais de 10 s...")
    del sinais
    gc.collect()
else:
    print("Array 'sinais' já havia sido liberado (reexecução detectada).")

# Concatenação final
matriz_bt    = np.concatenate(lista_janelas, axis=0)
df_bt        = pd.DataFrame(lista_meta_bt)
del lista_janelas
gc.collect()

# Relatório de sobreposição de janelas
n_sobreposicao = df_bt[df_bt['aviso_sobreposicao'] == True]['ecg_id'].nunique()
pct_sobreposicao = 100 * n_sobreposicao / df_bt['ecg_id'].nunique()

display(Markdown(f"""
**Extração concluída:**
- Batimentos extraídos: **{len(matriz_bt):,}**
- Shape da matriz: **{matriz_bt.shape}** (batimentos × amostras × derivações)
- Registros sem batimentos detectados: **{len(df_instancias) - df_bt['ecg_id'].nunique()}**
- Registros com risco de sobreposição de janelas (RR mín < 600 ms): **{n_sobreposicao} ({pct_sobreposicao:.1f}%)**
"""))

NameError: name 'df_instancias' is not defined

**Comentários sobre a subseção 3.3:**

[Inserir comentário após execução — reportar o número total de batimentos extraídos, a média de batimentos por registro (que deve ser compatível com a faixa de 6–15 batimentos em 10 s para frequências cardíacas de 60–100 bpm), e o percentual de registros em que o detector não encontrou nenhum pico válido. Verificar se registros sem detecção correspondem predominantemente às classes diagnósticas com maior irregularidade de ritmo (ex: AFIB).]

---

---
---
## Seção 4 — Validação da Estratégia de Segmentação

A validação da segmentação ocorre em dois eixos complementares, conforme exigido pelo pipeline:

- **Estabilidade intra-janela:** avalia se as janelas individuais (10 s e 600 ms) não contêm sinais degenerados, avaliando métricas de variância e taxa de cruzamento por zero (ZCR);
- **Variância inter-janela:** avalia se batimentos do mesmo registro são consistentes entre si, e se batimentos de classes diferentes apresentam variabilidade estatisticamente distinta.

Essas análises são essenciais para garantir que o dataset de instâncias produzido por esta etapa é adequado para a extração de features no Entregável 6.

---

### 4.1 Estabilidade Intra-Janela (Nível Registro — 10 s)

Para verificar que os registros de 10 s não contêm sinais mortos ou com artefatos residuais, calculamos duas métricas por registro na derivação DII:

- **ZCR (Zero Crossing Rate):** proporção de amostras em que o sinal muda de sinal. Sinais fisiológicos típicos apresentam ZCR moderada; valores extremos (muito baixo = sinal constante; muito alto = ruído de alta frequência) indicam problemas.

$$ZCR = \frac{1}{N-1} \sum_{n=1}^{N-1} \mathbb{1}[s_n \cdot s_{n-1} < 0]$$

- **Variância ($\sigma^2$):** mede a dispersão das amplitudes ao longo do registro. Valores próximos de zero indicam sinais planos (saturação ou eletrodo solto).

Para eficiência de memória, os sinais de 10 s são recarregados via `mmap_mode='r'`, evitando alocação total em RAM.

---

In [ ]:
# Recarrega via mmap — seguro mesmo se sinais foi deletado na seção 3
sinais_mmap = np.load(str(DIR_OUT_D4 / 'sinais_bons_100hz.npy'), mmap_mode='r')

n_total = len(df_instancias)
#Por eficiência computacional, a análise é realizada sobre uma amostra aleatória de 50% dos registros
amostra_idx = np.random.choice(n_total, size=min(int(n_total * 0.50), 5000), replace=False)

zcr_vals, var_vals, eid_vals = [], [], []

for idx in tqdm(amostra_idx, desc='Calculando estabilidade intra-janela'):
    lead_ii = sinais_mmap[idx, :, 1]

    # ZCR
    mudancas_sinal = np.diff(np.sign(lead_ii)) != 0
    zcr = np.sum(mudancas_sinal) / (len(lead_ii) - 1)

    zcr_vals.append(zcr)
    var_vals.append(np.var(lead_ii))
    eid_vals.append(df_instancias.index[idx])

del sinais_mmap
gc.collect()

df_stab = pd.DataFrame({'ecg_id': eid_vals, 'zcr': zcr_vals, 'variancia': var_vals})
df_stab = df_stab.merge(
    df_instancias[['primary_class']].reset_index(),
    on='ecg_id', how='left'
)

# Limites críticos
zcr_low_lim   = df_stab['zcr'].quantile(0.01)
var_low_lim   = 1e-4

n_zcr_baixo   = (df_stab['zcr'] < zcr_low_lim).sum()
n_var_baixa   = (df_stab['variancia'] < var_low_lim).sum()

display(Markdown(f"""
**Métricas de Estabilidade Intra-Janela (amostra de {len(df_stab)} registros):**

| Métrica | Média | Mediana | Mín | Máx |
|---|---|---|---|---|
| ZCR | {df_stab['zcr'].mean():.4f} | {df_stab['zcr'].median():.4f} | {df_stab['zcr'].min():.4f} | {df_stab['zcr'].max():.4f} |
| Variância | {df_stab['variancia'].mean():.5f} | {df_stab['variancia'].median():.5f} | {df_stab['variancia'].min():.5f} | {df_stab['variancia'].max():.5f} |

- Registros com ZCR abaixo do percentil 1% (possível sinal plano): **{n_zcr_baixo}**
- Registros com variância < {var_low_lim} (possível saturação): **{n_var_baixa}**
"""))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# ZCR por superclasse
sns.boxplot(
    data=df_stab,
    x='primary_class',
    y='zcr',
    hue='primary_class',
    order=['NORM', 'MI', 'CD', 'STTC', 'HYP'],
    palette='viridis',
    legend=False,
    ax=axes[0]
)
axes[0].set_title('Zero Crossing Rate por Superclasse (Registro 10 s)', fontweight='bold')
axes[0].set_xlabel('Superclasse')
axes[0].set_ylabel('ZCR')

# Variância por superclasse
sns.boxplot(
    data=df_stab,
    x='primary_class',
    y='variancia',
    hue='primary_class',
    order=['NORM', 'MI', 'CD', 'STTC', 'HYP'],
    palette='rocket',
    legend=False,
    ax=axes[1]
)
axes[1].set_title('Variância por Superclasse (Registro 10 s)', fontweight='bold')
axes[1].set_xlabel('Superclasse')
axes[1].set_ylabel('Variância (mV²)')

plt.tight_layout()
fig.savefig(FIGS_DIR / 'estabilidade_intra_janela.png', dpi=150, bbox_inches='tight')
plt.show()

**Comentários sobre a subseção 4.1:**

[Inserir comentário após análise dos gráficos — verificar se os registros apresentam ZCR e variância dentro de faixas esperadas para sinais de ECG fisiológicos (ZCR típica entre 0.05 e 0.30; variância acima de 1×10⁻³ mV²). Comparar os padrões entre superclasses e verificar se há consistência com os achados de amplitude dos Entregáveis 3 e 4. Reportar se os limiares críticos identificaram registros problemáticos residuais.]

---

### 4.2 Variância Inter-Janela (Nível Batimento)

A análise de variância inter-janela avalia a **consistência entre batimentos** dentro de um mesmo registro e entre registros de diferentes classes diagnósticas. A ideia central é que batimentos extraídos de um mesmo registro devem apresentar morfologia similar (baixa variância intra-registro), enquanto batimentos de classes distintas devem apresentar variabilidade morfológica mais elevada (alta variância inter-classe).

Para cada registro, calculamos:

- **Variância intra-registro:** variância da amplitude média dos batimentos do mesmo ECG na derivação DII
- **Amplitude pico a pico média:** pico a pico médio dos batimentos do registro (proxy de consistência morfológica)

Esses valores são então comparados entre as superclasses por meio de boxplots e estatísticas descritivas.

---

In [ ]:
# Métricas por batimento (derivação DII = índice 1)
lead_ii_beats      = matriz_bt[:, :, 1]                          # (N_beats × 60)
df_bt['amp_rms']   = np.sqrt(np.mean(lead_ii_beats ** 2, axis=1))
df_bt['amp_p2p']   = np.ptp(lead_ii_beats, axis=1)
df_bt['variancia'] = np.var(lead_ii_beats, axis=1)

# Agregação por registro
df_inter = df_bt.groupby('ecg_id').agg(
    n_beats          = ('rr_interval_ms', 'count'),
    rr_mean_ms       = ('rr_interval_ms', 'mean'),
    rr_std_ms        = ('rr_interval_ms', 'std'),
    amp_rms_mean     = ('amp_rms',   'mean'),
    amp_p2p_mean     = ('amp_p2p',   'mean'),
    var_intra_beat   = ('amp_rms',   'std'),   # variabilidade do RMS entre batimentos
    primary_class    = ('primary_class', 'first')
).reset_index()

# Filtra registros com pelo menos 3 batimentos
df_inter = df_inter[df_inter['n_beats'] >= 3]

display(Markdown("**Estatísticas inter-janela por superclasse:**"))
display(df_inter.groupby('primary_class')[['n_beats', 'rr_mean_ms', 'rr_std_ms',
                                            'var_intra_beat']].median().round(2))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

ordens_cls = ['NORM', 'MI', 'CD', 'STTC', 'HYP']

# Variância intra-batimento
sns.boxplot(data=df_inter, x='primary_class', y='var_intra_beat',
            hue='primary_class', order=ordens_cls, palette='mako',
            legend=False, ax=axes[0])
axes[0].set_title('Variância Inter-Janela\n(Variabilidade do RMS entre batimentos)', fontweight='bold')
axes[0].set_xlabel('Superclasse')
axes[0].set_ylabel('STD do RMS (mV)')

# RR std (irregularidade de ritmo)
sns.boxplot(data=df_inter, x='primary_class', y='rr_std_ms',
            hue='primary_class', order=ordens_cls, palette='viridis',
            legend=False, ax=axes[1])
axes[1].set_title('Variabilidade do Intervalo RR\n(Irregularidade de Ritmo)', fontweight='bold')
axes[1].set_xlabel('Superclasse')
axes[1].set_ylabel('SDNN (ms)')

# Número de batimentos por registro
sns.boxplot(data=df_inter, x='primary_class', y='n_beats',
            hue='primary_class', order=ordens_cls, palette='crest',
            legend=False, ax=axes[2])
axes[2].set_title('Número de Batimentos por Registro', fontweight='bold')
axes[2].set_xlabel('Superclasse')
axes[2].set_ylabel('N° Batimentos')

plt.suptitle('Análise de Variância Inter-Janela por Superclasse', fontsize=14, fontweight='bold')
plt.tight_layout()
fig.savefig(FIGS_DIR / 'variancia_inter_janela.png', dpi=150, bbox_inches='tight')
plt.show()

**Comentários sobre a subseção 4.2:**

[Inserir comentário após análise dos gráficos — verificar se a variância inter-janela (variabilidade do RMS entre batimentos) é consistente com o esperado fisiologicamente: ritmos irregulares (ex: AFIB, capturado em CD) devem apresentar maior SDNN; registros normais devem exibir maior consistência morfológica entre batimentos (menor var_intra_beat). Reportar se o número médio de batimentos por registro é compatível com a faixa de frequência cardíaca esperada para cada classe.]

---

### 4.3 Validação Visual: Sobreposição de Batimentos por Classe

A sobreposição de todos os batimentos extraídos de um registro, centrados no pico R, permite verificar visualmente:

- a precisão do alinhamento promovido pelo detector Pan-Tompkins (batimentos bem alinhados mostram convergência clara em torno do pico);
- a morfologia típica de cada superclasse diagnóstica no nível do batimento individual.

O batimento médio (calculado como a média amostral entre todos os batimentos do registro) é destacado em vermelho, permitindo comparar o "template" do registro com a variabilidade intra-registro.

---

In [ ]:
# Eixo exato com base nas constantes já definidas
pre  = int(0.20 * FS)   # 20 amostras
post = int(0.40 * FS)   # 40 amostras
t_beat_ms = np.arange(-pre, post) * (1000 / FS)
# resultado: [-200, -190, ..., -10, 0, 10, ..., 390] — 60 pontos, espaçamento exato de 10 ms

fig, axes = plt.subplots(1, len(ordens_cls), figsize=(20, 5))

for i, cls in enumerate(ordens_cls):
    ax = axes[i]

    # Seleciona um registro representativo da classe
    eids_cls = df_inter[df_inter['primary_class'] == cls]['ecg_id'].values
    if len(eids_cls) == 0:
        ax.set_title(f'{cls}\n(sem dados)')
        continue

    eid_ref = eids_cls[0]

    # Índices dos batimentos deste registro na matriz
    idx_beats = df_bt[df_bt['ecg_id'] == eid_ref].index.tolist()
    beats_cls = matriz_bt[idx_beats, :, 1]   # Lead II

    for b in beats_cls:
        ax.plot(t_beat_ms, b, color='#3498db', alpha=0.2, lw=0.8)

    ax.plot(t_beat_ms, np.mean(beats_cls, axis=0),
            color='#e74c3c', lw=2.5, label=f'Média (n={len(beats_cls)})')

    ax.axvline(0, color='gray', linestyle='--', alpha=0.6, label='Pico R')
    ax.set_title(f'{cls} — ID {eid_ref}', fontweight='bold')
    ax.set_xlabel('Tempo (ms) relativo ao pico R')
    if i == 0:
        ax.set_ylabel('Amplitude DII (mV)')
    ax.legend(fontsize=8)

plt.suptitle('Sobreposição de Batimentos por Superclasse (Lead DII)', fontsize=13)
plt.tight_layout()
fig.savefig(FIGS_DIR / 'sobreposicao_beats_por_classe.png', dpi=150, bbox_inches='tight')
plt.show()

NameError: name 'ordens_cls' is not defined

**Comentários sobre a subseção 4.3:**

[Inserir comentário após análise dos gráficos — verificar se o alinhamento pelo pico R é visualmente preciso (convergência dos batimentos próximo ao tempo 0 ms) e se os batimentos médios de cada classe apresentam as morfologias esperadas com base no conhecimento clínico (ex: HYP com maior amplitude, MI com possível elevação/depressão de ST, NORM com morfologia limpa e estável). Comentar sobre a dispersão dos batimentos individuais como indicativo da variabilidade intra-registro de cada classe.]

---

### 4.4 Diagrama de Poincaré dos Intervalos RR

O **diagrama de Poincaré** é uma ferramenta não linear de análise da variabilidade de frequência cardíaca (HRV), amplamente utilizada na literatura de processamento de biossinais. Ele representa graficamente cada intervalo RR em função do intervalo imediatamente anterior:

$$\text{Plot: } (RR_n, RR_{n+1}), \quad n = 1, 2, \ldots, N-1$$

A forma geométrica resultante revela padrões de regularidade e irregularidade no ritmo cardíaco:

- **Nuvem elipsoidal estreita e compacta:** ritmo regular (ex: NORM, sinusal)
- **Nuvem elipsoidal larga:** grande variabilidade, ritmo irregular ou com arritmia
- **Padrão difuso sem estrutura:** fibrilação atrial ou outras arritmias complexas

Essa visualização funciona como uma **validação qualitativa** do detector Pan-Tompkins: se os picos R foram detectados corretamente, o diagrama de Poincaré deve exibir padrões fisiológicos coerentes com a classe diagnóstica do registro.

---

In [ ]:
fig, axes = plt.subplots(1, len(ordens_cls), figsize=(20, 4))
cores_cls = ['#2ecc71', '#e74c3c', '#3498db', '#f39c12', '#9b59b6']

for i, cls in enumerate(ordens_cls):
    ax = axes[i]

    # Pega todos os RR desta classe (amostra dos registros)
    eids_cls = df_inter[df_inter['primary_class'] == cls]['ecg_id'].values[:20]

    rr_n, rr_n1 = [], []
    for eid in eids_cls:
        rr_vals = df_bt[df_bt['ecg_id'] == eid]['rr_interval_ms'].dropna().values
        if len(rr_vals) >= 2:
            rr_n.extend(rr_vals[:-1])
            rr_n1.extend(rr_vals[1:])

    ax.scatter(rr_n, rr_n1, color=cores_cls[i], alpha=0.3, s=10)
    ax.set_title(f'{cls}', fontweight='bold')
    ax.set_xlabel('$RR_n$ (ms)')
    if i == 0:
        ax.set_ylabel('$RR_{n+1}$ (ms)')

    # Linha de identidade
    lim = [min(rr_n + rr_n1), max(rr_n + rr_n1)] if rr_n else [0, 1000]
    ax.plot(lim, lim, 'k--', alpha=0.4, lw=0.8)

plt.suptitle('Diagrama de Poincaré por Superclasse (Intervalos RR)', fontsize=13)
plt.tight_layout()
fig.savefig(FIGS_DIR / 'poincare_rr_por_classe.png', dpi=150, bbox_inches='tight')
plt.show()

**Comentários sobre a subseção 4.4:**

[Inserir comentário após análise dos gráficos — descrever o padrão geométrico observado para cada classe. NORM deve exibir elipse compacta (baixa variabilidade); CD, que inclui distúrbios de condução como AFIB, deve apresentar nuvem mais dispersa. Verificar se os padrões são coerentes com o conhecimento clínico e se validam a qualidade da detecção de picos R pelo algoritmo Pan-Tompkins.]

---

### 4.5 Teste Estatístico: Diferença de Variância Inter-Classe (Kruskal-Wallis)

Para complementar a análise visual, aplicamos o **teste de Kruskal-Wallis** — alternativa não paramétrica à ANOVA — para verificar se existe diferença estatisticamente significativa na variância inter-janela (variabilidade do RMS entre batimentos) entre as superclasses diagnósticas.

A escolha pelo teste não paramétrico é motivada pelos resultados do Entregável 3, que demonstraram que as métricas de sinal **não seguem distribuição normal** e apresentam **heterocedasticidade generalizada**, tornando a ANOVA clássica inadequada.

**Hipóteses:**
- $H_0$: as distribuições de variância inter-janela são iguais entre as classes
- $H_1$: pelo menos uma classe apresenta distribuição diferente

Em caso de rejeição de $H_0$ (p < 0.05), o teste indica que as classes diagnósticas apresentam diferenças detectáveis na consistência morfológica dos batimentos, o que confere poder discriminativo potencial para as features extraídas no Entregável 6.

---

In [ ]:
ordens_test = ['NORM', 'MI', 'CD', 'STTC', 'HYP']

grupos_variancia = [
    df_inter[df_inter['primary_class'] == cls]['var_intra_beat'].dropna().values
    for cls in ordens_test
    if len(df_inter[df_inter['primary_class'] == cls]) >= 5
]

grupos_rr_std = [
    df_inter[df_inter['primary_class'] == cls]['rr_std_ms'].dropna().values
    for cls in ordens_test
    if len(df_inter[df_inter['primary_class'] == cls]) >= 5
]

if len(grupos_variancia) >= 2:
    stat_kw_var, p_kw_var = stats.kruskal(*grupos_variancia)
else:
    stat_kw_var, p_kw_var = np.nan, np.nan

if len(grupos_rr_std) >= 2:
    stat_kw_rr, p_kw_rr = stats.kruskal(*grupos_rr_std)
else:
    stat_kw_rr, p_kw_rr = np.nan, np.nan

display(Markdown(f"""
### Resultados — Kruskal-Wallis (5 superclasses)

| Métrica | Estatística H | p-valor | Rejeita H₀? |
|---|---|---|---|
| Variância inter-batimento (var_intra_beat) | {stat_kw_var:.4f} | {'< 0.001' if p_kw_var < 0.001 else round(p_kw_var, 4)} | {'**Sim** (p < 0.05)' if p_kw_var < 0.05 else 'Não'} |
| Desvio padrão RR (SDNN) | {stat_kw_rr:.4f} | {'< 0.001' if p_kw_rr < 0.001 else round(p_kw_rr, 4)} | {'**Sim** (p < 0.05)' if p_kw_rr < 0.05 else 'Não'} |
"""))

**Comentários sobre a subseção 4.5:**

[Inserir comentário após execução — interpretar os p-valores obtidos. Se H₀ for rejeitada, isso indica que as métricas de variabilidade inter-janela têm potencial discriminativo entre classes, o que é um resultado positivo para a fase de extração de features. Discutir quais pares de classes provavelmente apresentam as maiores diferenças com base nos boxplots da subseção 4.2.]

---

---
---
## Seção 5 — Exportação dos Artefatos e Síntese

Nesta seção final, consolidamos os dois níveis de segmentação e salvamos os artefatos necessários para o Entregável 6 (Extração de Atributos).

---

### 5.1 Estrutura Final dos Artefatos

O produto desta etapa é composto por dois níveis complementares:

| Artefato | Formato | Dimensão | Finalidade no E6 |
|---|---|---|---|
| `registros_ids.csv` | CSV | N_registros × M_colunas | Índice de instâncias, splits e labels |
| `batimentos_segmentados.npy` | NumPy | N_beats × 60 × 12 | Features morfológicas (domínio do tempo e frequência) |
| `batimentos_ids.csv` | CSV | N_beats × K_colunas | Mapeamento beat → ecg_id, classe, split |
| `metricas_inter_janela.csv` | CSV | N_registros × P_colunas | Métricas de HRV e variabilidade já calculadas |

---

In [ ]:
# 1. Salvar metadados das instâncias de 10 s
df_instancias.to_csv(OUT_DIR / 'registros_ids.csv')

# 2. Salvar matriz de batimentos
np.save(str(OUT_DIR / 'batimentos_segmentados.npy'), matriz_bt)

# 3. Salvar metadados de batimentos
df_bt.to_csv(OUT_DIR / 'batimentos_ids.csv', index=False)

# 4. Salvar métricas inter-janela por registro
df_inter.to_csv(OUT_DIR / 'metricas_inter_janela.csv', index=False)

# Resumo de salvamento
display(Markdown(f"""
### Exportação Concluída

| Arquivo | Registros / Shape |
|---|---|
| `registros_ids.csv` | {len(df_instancias)} instâncias de 10 s |
| `batimentos_segmentados.npy` | {matriz_bt.shape} |
| `batimentos_ids.csv` | {len(df_bt):,} batimentos |
| `metricas_inter_janela.csv` | {len(df_inter)} registros com métricas de ritmo |
"""))

### 5.2 Síntese e Conexão

Ao longo deste entregável, foi construída e validada a **estratégia de segmentação** dos sinais de ECG, estabelecendo a ponte entre a etapa de limpeza (Entregável 4) e a extração de atributos (Entregável 6).

---

#### Estratégia adotada

A decisão central desta etapa foi a adoção de uma **estratégia híbrida em dois níveis**. No nível primário, cada registro de 10 segundos constituiu uma instância independente, preservando a unidade clínica original do PTB-XL e garantindo a independência estatística entre amostras — pressuposto fundamental para os modelos de classificação do Entregável 10. No nível secundário, a segmentação por batimento via detector Pan-Tompkins permitiu acessar informações morfológicas finas (ondas P, QRS, T) e métricas de variabilidade de intervalo RR que não podem ser extraídas diretamente do registro completo.

Essa arquitetura evita os dois principais riscos da segmentação com sobreposição em pipelines de ML clássico: a **correlação artificial entre instâncias** e o **label inflation**, que infla artificialmente o número de amostras sem acrescentar informação independente real.

---

#### Validação da segmentação

A análise de **estabilidade intra-janela** confirmou que os registros processados apresentam ZCR e variância dentro das faixas esperadas para sinais fisiológicos de ECG, sem sinais planos ou saturados residuais — resultado coerente com a eficácia da pipeline de limpeza do Entregável 4. A análise de **variância inter-janela** evidenciou diferenças estatisticamente significativas entre as superclasses diagnósticas tanto na variabilidade morfológica dos batimentos quanto na irregularidade do ritmo (SDNN), o que confere poder discriminativo potencial às features que serão extraídas.

A visualização por sobreposição de batimentos e os diagramas de Poincaré validaram qualitativamente a precisão do detector Pan-Tompkins: o alinhamento dos batimentos em torno do pico R foi satisfatório, e os padrões de variabilidade de RR foram coerentes com o conhecimento clínico de cada classe — ritmos regulares em NORM e maior irregularidade em registros com distúrbios de condução.

(Texto base, alterar conforme os resultados)

---

#### Produto entregue e conexão com o Entregável 6

Os artefatos gerados estruturam o pipeline em dois fluxos paralelos:

1. **Fluxo de features globais** — alimentado pelos registros de 10 s (`sinais_bons_100hz.npy`), sobre os quais serão calculadas features de domínio do tempo (RMS, variância), frequência (PSD por banda) e não lineares (entropia, DFA);

2. **Fluxo de features morfológicas** — alimentado pela matriz de batimentos (`batimentos_segmentados.npy`), sobre a qual serão calculadas features de forma (duração do QRS, amplitude das ondas) e as métricas de HRV já parcialmente calculadas neste entregável (`metricas_inter_janela.csv`).

Ao final do Entregável 6, esses dois fluxos serão consolidados em um único vetor de features por registro, que servirá de entrada para as etapas de engenharia de features, redução de dimensionalidade e seleção de atributos. 

---